In [1]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
import random
import pickle

In [ ]:
def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

set_seed(SEED)

device = torch.device("cuda")

In [3]:
class MusicDataset(Dataset):
    def __init__(self):
        with open("pakiet/train.pkl", "rb") as file:
            train_raw = pickle.load(file)
        self.xx = []
        self.yy = []
        for x, y in train_raw:
            self.xx.append(torch.tensor(x, dtype=torch.float32))
            self.yy.append(torch.tensor(y, dtype=torch.long).unsqueeze(0))

    def __len__(self):
        return len(self.xx)

    def __getitem__(self, idx):
        return self.xx[idx], self.yy[idx]

In [4]:
pad_value = 0


def pad_collate(batch):
    xx, yy = zip(*batch)
    yy = torch.stack(yy)
    x_lens = [x.shape[0] for x in xx]
    y_lens = [1] * len(yy)

    xx_pad = pad_sequence(xx, batch_first=True, padding_value=pad_value)

    return xx_pad, yy, x_lens, y_lens

In [5]:
full_trainset = MusicDataset()

train_ratio = 0.8

targets = full_trainset.yy
indices = list(range(len(full_trainset)))

train_indices, val_indices = train_test_split(
    indices, train_size=train_ratio, stratify=targets, random_state=SEED
)

trainset = Subset(full_trainset, train_indices)
valset = Subset(full_trainset, val_indices)

# num_classes = len(trainset.dataset.classes)

# print(f"Number of classes: {num_classes}")
print(f"Trainset size: {len(trainset)}")
print(f"Validation set size: {len(valset)}")

Trainset size: 2351
Validation set size: 588


In [6]:
batch_size = 64
num_workers = 0
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    drop_last=True,
    collate_fn=pad_collate,
)
valloader = torch.utils.data.DataLoader(
    valset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    collate_fn=pad_collate,
)

In [7]:
class LSTMClassifier(nn.Module):

    def __init__(
        self,
        input_size,
        hidden_size,
        num_layers,
        out_size,
        dropout_prob=0.3,
        vocab_size=182,
        embedding_dim=64,
    ):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout_prob if num_layers > 1 else 0,
        )

        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(hidden_size * 2, out_size)

    def init_hidden(self, batch_size):
        hidden = torch.zeros(self.num_layers * 2, batch_size, self.hidden_size)
        state = torch.zeros(self.num_layers * 2, batch_size, self.hidden_size)
        return hidden, state

    def forward(self, x_padded, x_len, hidden):
        x_embedded = self.embedding(x_padded+1)
        x_packed = nn.utils.rnn.pack_padded_sequence(
            x_embedded, x_len, batch_first=True, enforce_sorted=False
        )

        packed_outputs, (h_n, c_n) = self.lstm(x_packed, hidden)

        forward_hidden = h_n[-2]
        backward_hidden = h_n[-1]

        last_out = torch.cat((forward_hidden, backward_hidden), dim=1)

        out = self.dropout(last_out)
        out = self.fc(out)

        return out

In [8]:
all_x = torch.cat([trainset[i][0].long() for i in range(len(trainset))])
vocab_size = all_x.max() + 2
vocab_size

tensor(193)

In [ ]:
net = LSTMClassifier(
    input_size=1,
    hidden_size=256,
    num_layers=3,
    out_size=5,
    dropout_prob=0.5,
    vocab_size=vocab_size,
)

In [ ]:
epochs = 40
optimizer = torch.optim.AdamW(net.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=0.01,
    steps_per_epoch=len(trainloader),
    epochs=epochs,
)
loss_fun = nn.CrossEntropyLoss()
net.train()
net.to(device)

# Training loop
for epoch in range(epochs):
    for x, targets, x_len, target_len in trainloader:
        x = x.to(device).long()
        targets = targets.to(device).squeeze(-1)
        hidden, state = net.init_hidden(x.size(0))
        hidden, state = hidden.to(device), state.to(device)

        preds = net(x, x_len, (hidden, state))

        optimizer.zero_grad()
        loss = loss_fun(preds, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
    print(f"Epoch: {epoch}, loss: {loss.item():.3}")

Epoch: 0, loss: 1.05
Epoch: 1, loss: 0.92
Epoch: 2, loss: 0.694
Epoch: 3, loss: 0.772
Epoch: 4, loss: 0.775
Epoch: 5, loss: 0.573
Epoch: 6, loss: 0.743
Epoch: 7, loss: 0.7
Epoch: 8, loss: 0.679
Epoch: 9, loss: 0.682
Epoch: 10, loss: 0.634
Epoch: 11, loss: 0.599
Epoch: 12, loss: 0.436
Epoch: 13, loss: 0.647
Epoch: 14, loss: 0.434
Epoch: 15, loss: 0.437
Epoch: 16, loss: 0.405
Epoch: 17, loss: 0.488
Epoch: 18, loss: 0.544
Epoch: 19, loss: 0.221
Epoch: 20, loss: 0.4
Epoch: 21, loss: 0.218
Epoch: 22, loss: 0.292
Epoch: 23, loss: 0.228
Epoch: 24, loss: 0.363
Epoch: 25, loss: 0.181
Epoch: 26, loss: 0.164
Epoch: 27, loss: 0.179
Epoch: 28, loss: 0.0661
Epoch: 29, loss: 0.185
Epoch: 30, loss: 0.122
Epoch: 31, loss: 0.04
Epoch: 32, loss: 0.151
Epoch: 33, loss: 0.0822
Epoch: 34, loss: 0.131
Epoch: 35, loss: 0.0416
Epoch: 36, loss: 0.0963
Epoch: 37, loss: 0.00433
Epoch: 38, loss: 0.0557
Epoch: 39, loss: 0.115


In [11]:
# # loading
# PATH = "net2.pth"
#
# net.load_state_dict(torch.load(PATH, map_location=device))
# net.to(device)
# net.eval()

In [12]:
# saving
PATH = "net5.pth"

torch.save(net.state_dict(), PATH)
print(f"Model weights saved to {PATH}")

Model weights saved to net5.pth


In [ ]:
def evaluate_accuracy(model, dataloader, device):
    model.eval()
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for x, targets, x_len, target_len in dataloader:
            x = x.to(device).long()
            targets = targets.to(device).squeeze(-1).long()  # Match shape [batch_size]

            hidden, state = model.init_hidden(x.size(0))
            hidden, state = hidden.to(device), state.to(device)

            preds = model(x,x_len, (hidden, state))

            pred_classes = torch.argmax(preds, dim=1)

            correct_predictions += (pred_classes == targets).sum().item()
            total_samples += targets.size(0)

    accuracy = correct_predictions / total_samples
    return accuracy

In [14]:
evaluate_accuracy(net, trainloader, device)

0.9926215277777778

In [15]:
evaluate_accuracy(net, valloader, device)

0.8979591836734694

In [ ]:
class TestMusicDataset(Dataset):
    def __init__(self, filepath="pakiet/test_no_target.pkl"):
        with open(filepath, "rb") as file:
            test_raw = pickle.load(file)
        
        self.xx = []
        for x in test_raw:
             self.xx.append(torch.tensor(x, dtype=torch.float32))

    def __len__(self):
        return len(self.xx)

    def __getitem__(self, idx):
        return self.xx[idx]
    
def test_pad_collate(batch):
    xx = batch
    x_lens = [x.shape[0] for x in xx]
    xx_pad = pad_sequence(xx, batch_first=True, padding_value=pad_value)
    return xx_pad, x_lens

In [ ]:
testset = TestMusicDataset("pakiet/test_no_target.pkl")

testloader = DataLoader(
    testset,
    batch_size=batch_size,
    shuffle=False, 
    num_workers=num_workers,
    pin_memory=True,
    collate_fn=test_pad_collate,
)

In [18]:
def generate_test_predictions(model, dataloader, device):
    model.eval()
    all_preds = []

    with torch.no_grad():
        for x, x_len in dataloader:
            x = x.to(device).long()
            
            hidden, state = model.init_hidden(x.size(0))
            hidden, state = hidden.to(device), state.to(device)

            preds = model(x, x_len, (hidden, state))
            
            pred_classes = torch.argmax(preds, dim=1)
            all_preds.extend(pred_classes.cpu().numpy())

    return all_preds

In [ ]:
print("Generowanie predykcji...")
predictions = generate_test_predictions(net, testloader, device)

output_filename = "pred.csv"
df_preds = pd.DataFrame(predictions)

df_preds.to_csv(output_filename, index=False, header=False)
print(f"Predykcje zapisane do {output_filename}")

Generowanie predykcji...
Predykcje zapisane do pred.csv
